# DSEG660 – LoRA greeting convention on Qwen3-1.7B
Runs the full pipeline end to end on one GPU with a 16 GiB cap. The reported results used an Apple M3 Max (PyTorch MPS); on a machine with an NVIDIA GPU the scripts use CUDA instead. Each cell calls the script used for the reported results. See `README.md` for what each file does.

## 0. Environment
Create the venv once: `python3.12 -m venv .venv && . .venv/bin/activate && pip install -r requirements.lock.txt` (plus, for the GGUF step, llama.cpp cloned into `tools/llama.cpp` at commit `abca692` and `pip install ./tools/llama.cpp/gguf-py`; see `README.md`). Run this notebook with that kernel.

In [ ]:
import json, subprocess, sys
PY = sys.executable
def run(*args):
    print('$', ' '.join(args)); r = subprocess.run([PY, *args], capture_output=True, text=True); print(r.stdout[-3000:], r.stderr[-1500:] if r.returncode else '')
    r.check_returncode()

## 1. Build training and development data (deterministic, seed 3407)

In [ ]:
run('make_data.py')
print(json.load(open('data/exclusion_audit.json'))['n_reused_eval_sentences'], 'evaluation sentences reused')

## 2. Initial adapter: train (LR 2e-4, 294 steps, restart from base) and score on dev

In [ ]:
run('train.py', '--data', 'data/train_initial.jsonl', '--out', 'adapters/initial', '--lr', '2e-4', '--epochs', '1')
run('infer.py', '--data', 'data/dev.jsonl', '--adapter', 'adapters/initial', '--out', 'runs/initial_dev.raw.jsonl')
run('score.py', 'runs/initial_dev.raw.jsonl', 'data/dev.jsonl', '--decisions', 'runs/dev_decisions.json')
run('analyze_dev.py', 'runs/initial_dev.scored.jsonl', '--show', '10')

## 3. Targeted revision (position-word hard negatives) and final adapter (same recipe, restart from base)

In [ ]:
run('make_data.py', '--revise')
run('train.py', '--data', 'data/train_revised.jsonl', '--out', 'adapters/final', '--lr', '2e-4', '--epochs', '1')
run('infer.py', '--data', 'data/dev.jsonl', '--adapter', 'adapters/final', '--out', 'runs/final_dev.raw.jsonl')
run('score.py', 'runs/final_dev.raw.jsonl', 'data/dev.jsonl', '--decisions', 'runs/dev_decisions.json')

## 4. Evaluation set: base, initial, final (prompt only, chat template, non-thinking, greedy, 64 new tokens)

In [ ]:
for name in ['base', 'initial', 'final']:
    extra = [] if name == 'base' else ['--adapter', f'adapters/{name}']
    run('infer.py', '--data', 'data/source/dseg660-evaluation.jsonl', *extra, '--batch-size', '1', '--out', f'runs/{name}_benchmark.raw.jsonl')
    run('score.py', f'runs/{name}_benchmark.raw.jsonl', 'data/source/dseg660-evaluation.jsonl', '--decisions', 'runs/benchmark_decisions.json')
    print(name, json.load(open(f'runs/{name}_benchmark.summary.json'))['targets'])

## 5. Export Q4_K_M GGUF and run the eight deployment prompts in Ollama
Needs `llama-quantize` (`brew install llama.cpp` on macOS), Ollama, and `ollama serve` running in another terminal.

In [ ]:
run('export_gguf.py', '--adapter', 'adapters/final')
run('ollama_check.py', '--gguf', 'gguf/dseg660-qwen3-1.7b-Q4_K_M.gguf', '--adapter-raw', 'runs/final_benchmark.raw.jsonl')

## 6. Report

In [ ]:
run('report/build_report.py')